# Rubrics and Multi-Predictor Feedback

A single yes/no judge has biases and blind spots. This notebook covers two patterns that fix that: **criteria rubrics** that score multiple dimensions and combine them (with optional weights), and **multi-predictor feedback** that uses DSPy's `trace` to tell optimizers like GEPA exactly which step of a pipeline failed and why.

In [ ]:
%pip install -r ../requirements.txt -q

In [ ]:
import os
import dspy
from dotenv import load_dotenv

load_dotenv()

dspy.configure(lm=dspy.LM('openai/gpt-5-mini'))

## Criteria rubric

Define one signature that scores several dimensions at once, then combine the scores into a single metric value. The judge below grades marketing copy on clarity, persuasiveness, brand fit, and emotional appeal.

In [ ]:
class EvaluateMarketing(dspy.Signature):
    """Evaluate marketing copy on multiple dimensions."""
    copy: str = dspy.InputField(desc="Marketing copy to evaluate")

    # Multiple evaluation dimensions
    clarity: int = dspy.OutputField(desc="1-5: How clear is the message?")
    persuasiveness: int = dspy.OutputField(
        desc="1-5: How persuasive is the copy?"
    )
    brand_fit: int = dspy.OutputField(desc="1-5: Does it match brand voice?")
    emotional_appeal: int = dspy.OutputField(
        desc="1-5: Does it resonate emotionally?"
    )

    reasoning: str = dspy.OutputField(desc="Explain your scores")

### Average all criteria

The simplest combination is the unweighted mean. Normalize from the 1–5 range down to 0–1 so the metric plays well with DSPy's evaluator and optimizers.

In [ ]:
def marketing_rubric_metric(example, pred, trace=None):
    evaluator = dspy.ChainOfThought(EvaluateMarketing)
    result = evaluator(copy=pred.marketing_copy)

    # Average across criteria
    avg_score = (
        result.clarity
        + result.persuasiveness
        + result.brand_fit
        + result.emotional_appeal
    ) / 4.0

    # Normalize to 0-1 scale
    normalized = (avg_score - 1) / 4.0

    return normalized


example = dspy.Example(brief="Launch ad for an AI productivity app")
pred = dspy.Prediction(
    marketing_copy=(
        "Stop drowning in busywork. Our AI co-pilot drafts the emails, "
        "schedules the meetings, and surfaces what actually matters — so "
        "you can ship the work only you can do."
    )
)

print(marketing_rubric_metric(example, pred))

### Weighted scoring

Some dimensions matter more than others. Use a weighted average to reflect business value — work with your domain experts to set weights and sanity-check that your best and worst examples score the way you'd expect.

In [ ]:
def weighted_marketing_metric(example, pred, trace=None):
    evaluator = dspy.ChainOfThought(EvaluateMarketing)
    result = evaluator(copy=pred.marketing_copy)

    # Define weights (must sum to 1.0)
    weights = {
        "clarity": 0.3,
        "persuasiveness": 0.4,  # Most important
        "brand_fit": 0.2,
        "emotional_appeal": 0.1,
    }

    # Weighted average
    weighted_score = (
        result.clarity * weights["clarity"]
        + result.persuasiveness * weights["persuasiveness"]
        + result.brand_fit * weights["brand_fit"]
        + result.emotional_appeal * weights["emotional_appeal"]
    )

    # Normalize to 0-1
    normalized = (weighted_score - 1) / 4.0

    return normalized


print(weighted_marketing_metric(example, pred))

## Multi-predictor feedback with `trace`

Most DSPy programs are pipelines: one predictor generates a search query, another retrieves passages, a third synthesizes the answer. A metric that only looks at the final answer misses *which* predictor failed. During optimization DSPy passes the full call trace into your metric as the `trace` argument — a list of `(predictor, inputs, outputs)` tuples, where the first element is the predictor object itself.

Below is a two-hop QA module — the kind of pipeline that benefits most from per-predictor feedback.

In [ ]:
class MultiHopQA(dspy.Module):
    def __init__(self):
        super().__init__()
        self.generate_query = dspy.ChainOfThought("question -> query")
        self.retrieve = dspy.Retrieve(k=3)
        self.synthesize = dspy.ChainOfThought(
            "question, context -> answer"
        )

    def forward(self, question):
        # Step 1: Generate search query
        query_result = self.generate_query(question=question)

        # Step 2: Retrieve passages (.passages is required — Retrieve returns
        # a Prediction object, not a bare list)
        passages = self.retrieve(query_result.query).passages

        # Step 3: Synthesize answer
        context = "\n".join(passages)
        answer_result = self.synthesize(
            question=question,
            context=context,
        )

        return dspy.Prediction(
            query=query_result.query,
            answer=answer_result.answer,
        )

### A trace-aware metric

When `trace` is `None`, the metric is being used for plain evaluation — return a simple score. When `trace` is populated, the metric is running inside an optimizer like GEPA — return a `dspy.Prediction(score=..., feedback=...)` with per-step diagnostics the optimizer can reflect on.

In [ ]:
def multi_hop_metric(example, pred, trace=None):
    # Check if answer is correct
    correct = example.answer.lower() in pred.answer.lower()

    if trace is None:
        return correct

    # Analyze intermediate steps.
    # Each trace element is (predictor_instance, inputs_dict, outputs) —
    # the first element is the predictor object itself, not a name string.
    feedback = []

    for predictor, inputs, outputs in trace:
        # Identify the query-generation step by its output field
        if hasattr(outputs, "query"):
            query = outputs.query
            # Check if query includes key entities from question
            question_words = set(inputs['question'].lower().split())
            query_words = set(query.lower().split())
            overlap = len(question_words & query_words)

            if overlap < 2:
                feedback.append(
                    f"Query '{query}' shares few words with question. "
                    "Preserve key entities when reformulating."
                )

    if not correct:
        feedback.append(
            f"Answer '{pred.answer}' is incorrect. "
            f"Expected: '{example.answer}'"
        )

    return dspy.Prediction(
        score=1.0 if correct else 0.0,
        feedback="\n".join(feedback) if feedback else "Correct!",
    )

### Per-predictor feedback dict

GEPA can also accept a `feedback` *dict* keyed by predictor name. That lets you send a different improvement hint to each predictor in the pipeline, instead of one big shared string.

In [ ]:
def pipeline_metric_with_feedback(example, pred, trace=None):
    # Check final answer correctness
    is_correct = pred.answer.lower().strip() == example.answer.lower().strip()

    if trace is None:
        return is_correct

    # Build per-predictor feedback for the optimizer.
    # Each trace element is (predictor_instance, inputs_dict, outputs).
    # Identify steps by the output fields they produce rather than by name string.
    feedback = {}

    for predictor, inputs, outputs in trace:
        if hasattr(outputs, "query"):
            query = outputs.query
            if len(query.split()) < 3:
                feedback["generate_query"] = (
                    f"Query '{query}' is too short. "
                    "Generate more specific, detailed queries."
                )
            elif example.answer.lower() not in query.lower():
                feedback["generate_query"] = (
                    f"Query '{query}' doesn't reference key terms. "
                    "Include relevant entities from the question."
                )

        elif hasattr(outputs, "answer") and not hasattr(outputs, "query"):
            if len(outputs.answer) < 50:
                feedback["synthesize"] = (
                    "Synthesis is too brief. "
                    "Extract more relevant information from retrieved passages."
                )

    score = 1.0 if is_correct else 0.0
    return dspy.Prediction(score=score, feedback=feedback)

Trace-aware metrics shine on multi-step pipelines, complex debugging, and GEPA optimization where the optimizer can reflect on *why* a step failed. Skip them for single-predictor programs or when evaluation latency matters more than feedback quality — trace analysis adds overhead on every call.